In [1]:
# Cell 1: Dependency Installation
!pip install easyocr opencv-python matplotlib pillow numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 56.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 972.1/972.1 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 16.2 MB/s eta 0:00:00


In [7]:
# Cell 2: Imports and Setup
import cv2
import easyocr
import numpy as np
import difflib
import datetime
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

# Initialize EasyOCR Reader (CPU mode enabled for universal compatibility)
reader = easyocr.Reader(['en'], gpu=False)

In [3]:
# Cell 3: Synthetic Test Frame Generator
def create_sample_frames():
    """Generates two blackboard images simulating a live updated lecture board."""
    # Frame 1: Initial lecture notes
    img1 = Image.new('RGB', (800, 500), color=(30, 45, 35))
    draw1 = ImageDraw.Draw(img1)
    draw1.text((50, 50), "Lecture 1: Physics Notes", fill=(255, 255, 255))
    draw1.text((50, 120), "E = m * c^2", fill=(255, 255, 255))
    draw1.text((50, 190), "Newton First Law: F = 0", fill=(255, 255, 255))
    img1.save("frame_01.png")

    # Frame 2: Added notes and equation update
    img2 = Image.new('RGB', (800, 500), color=(30, 45, 35))
    draw2 = ImageDraw.Draw(img2)
    draw2.text((50, 50), "Lecture 1: Physics Notes", fill=(255, 255, 255))
    draw2.text((50, 120), "E = m * c^2", fill=(255, 255, 255))
    draw2.text((50, 190), "Newton First Law: F = 0", fill=(255, 255, 255))
    # New content added below
    draw2.text((50, 260), "New Topic: Kinetic Energy", fill=(240, 240, 100))
    draw2.text((50, 330), "KE = 0.5 * m * v^2", fill=(240, 240, 100))
    img2.save("frame_02.png")

    print("Sample frames generated: 'frame_01.png' and 'frame_02.png'")

create_sample_frames()

Sample frames generated: 'frame_01.png' and 'frame_02.png'


In [4]:
# Cell 4: Visual Change Detection Core
def detect_visual_change(img_path1, img_path2, threshold_pct=2.0):
    """
    Computes visual frame difference between two images.
    Returns: (bool change_detected, float difference_percentage, numpy_diff_mask)
    """
    img1 = cv2.imread(img_path1)
    img2 = cv2.imread(img_path2)
    
    # Convert images to grayscale
    gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)
    
    # Apply Gaussian blur to reduce camera noise
    blur1 = cv2.GaussianBlur(gray1, (5, 5), 0)
    blur2 = cv2.GaussianBlur(gray2, (5, 5), 0)
    
    # Compute absolute frame difference
    diff = cv2.absdiff(blur1, blur2)
    _, thresh = cv2.threshold(diff, 25, 255, cv2.THRESH_BINARY)
    
    # Calculate altered pixel percentage relative to total frame
    changed_pixels = np.count_nonzero(thresh)
    total_pixels = thresh.size
    diff_percentage = (changed_pixels / total_pixels) * 100
    
    change_detected = diff_percentage >= threshold_pct
    return change_detected, diff_percentage, thresh

In [8]:
# Cell 5: OCR Extraction, Text Diffing & Text File Writer
def extract_text_from_image(img_path, confidence_threshold=0.3):
    """Extracts text lines and bounding boxes using EasyOCR."""
    results = reader.readtext(img_path)
    extracted_lines = []
    bounding_boxes = []
    
    for bbox, text, prob in results:
        if prob >= confidence_threshold:
            extracted_lines.append(text.strip())
            bounding_boxes.append((bbox, text))
            
    return extracted_lines, bounding_boxes

def compute_text_changes(previous_lines, current_lines):
    """Identifies newly added lines using difflib."""
    differ = difflib.Differ()
    diff_result = list(differ.compare(previous_lines, current_lines))
    
    added_text = [line[2:] for line in diff_result if line.startswith('+ ')]
    removed_text = [line[2:] for line in diff_result if line.startswith('- ')]
    unchanged_text = [line[2:] for line in diff_result if line.startswith('  ')]
    
    return added_text, removed_text, unchanged_text

def save_extracted_notes_to_file(added_lines, full_current_board, file_path="lecture_notes.txt"):
    """Appends extracted notes and delta changes into a text file."""
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    with open(file_path, "a", encoding="utf-8") as f:
        f.write(f"\n==================================================\n")
        f.write(f"BOARD UPDATE LOGGED AT: {timestamp}\n")
        f.write(f"==================================================\n\n")
        
        f.write("--- NEWLY DETECTED CONTENT ---\n")
        if added_lines:
            for line in added_lines:
                f.write(f"  + {line}\n")
        else:
            f.write("  (No new lines detected, visual shift only)\n")
            
        f.write("\n--- COMPLETE CURRENT BOARD SNAPSHOT ---\n")
        for line in full_current_board:
            f.write(f"  * {line}\n")
            
        f.write("\n\n")
        
    print(f"Successfully saved extracted text log to '{file_path}'")

In [10]:
# Cell 6: End-to-End Execution Pipeline
import os

def run_phase_one_pipeline(frame1_path, frame2_path, output_txt_file="lecture_notes.txt"):
    print("Step 1: Running Visual Change Detection...")
    has_changed, change_pct, diff_mask = detect_visual_change(frame1_path, frame2_path)
    
    print(f"Visual Frame Difference: {change_pct:.2f}%")
    
    if not has_changed:
        print("No significant change detected between frames. Skipping OCR & File Writing.")
        return
    
    print("Change Threshold Exceeded! Extracting Text from Images...")
    lines_frame1, _ = extract_text_from_image(frame1_path)
    lines_frame2, boxes_frame2 = extract_text_from_image(frame2_path)
    
    print("Computing Delta Differences...")
    added, removed, unchanged = compute_text_changes(lines_frame1, lines_frame2)
    
    print("Writing Extracted Data to File...")
    save_extracted_notes_to_file(added, lines_frame2, file_path=output_txt_file)
    
    # Render Output in Console
    print("\n" + "="*40)
    print("           DETECTION REPORT          ")
    print("="*40)
    print(f"Newly Added Lines ({len(added)}):")
    for line in added:
        print(f"   [+] {line}")
    print("="*40)

    # Plot Visual Analysis
    img_bgr = cv2.imread(frame2_path)
    for bbox, text in boxes_frame2:
        top_left = (int(bbox[0][0]), int(bbox[0][1]))
        bottom_right = (int(bbox[2][0]), int(bbox[2][1]))
        color = (0, 255, 0) if text in added else (255, 0, 0)
        cv2.rectangle(img_bgr, top_left, bottom_right, color, 2)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(cv2.cvtColor(cv2.imread(frame1_path), cv2.COLOR_BGR2RGB))
    axes[0].set_title("Frame 1 (Base)")
    
    axes[1].imshow(diff_mask, cmap='gray')
    axes[1].set_title(f"Visual Mask ({change_pct:.1f}% Diff)")
    
    axes[2].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    axes[2].set_title("Frame 2 (Bounding Boxes Applied)")
    
    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

# Run the pipeline
run_phase_one_pipeline("frame_01.png", "frame_02.png", output_txt_file="lecture_notes.txt")

# Read back and print the generated text file contents safely
print("\n--- GENERATED 'lecture_notes.txt' FILE CONTENT ---")
if os.path.exists("lecture_notes.txt"):
    with open("lecture_notes.txt", "r", encoding="utf-8") as f:
        print(f.read())
else:
    print("File 'lecture_notes.txt' does not exist yet (no threshold changes logged).")

Step 1: Running Visual Change Detection...
Visual Frame Difference: 0.31%
No significant change detected between frames. Skipping OCR & File Writing.

--- GENERATED 'lecture_notes.txt' FILE CONTENT ---
File 'lecture_notes.txt' does not exist yet (no threshold changes logged).
